# Clase 1 · El sancocho

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelvo yo en vivo durante la primera mitad de la clase. Ustedes lo siguen, lo corren y me interrumpen. En la segunda mitad cada uno resuelve el gemelo, *El precio de la luz*, con la misma técnica y otros datos.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Convertir una pregunta de la vida diaria en un problema de datos con métrica, explorarlo, comparar modelos de aprendizaje automático y de series de tiempo contra una línea base, y validar respetando el tiempo. |
| **Enfoque** | Ciencia de Datos. |
| **Resultados de aprendizaje** | 1 (formular el problema, los objetivos y la métrica), 3 (flujo reproducible y trazable), 4 (implementar y evaluar un modelo) y 5 (comunicar a quien no es técnico). |
| **Uso de IA** | Pueden pedirle a Claude que explique una línea de código o que escriba una gráfica. Deben poder explicar sin ayuda qué es una línea base, por qué no se barajan los meses, por qué el ganador se escoge en validación y qué significa el error que reportan. |
| **Entregable** | Este cuaderno corrido de principio a fin en su portafolio, con la decisión de la sección 4 escrita por ustedes. |

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path.cwd() / "taller-cd-ml-2026-2"] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

PALETA = ["#1D4E89", "#D7263D", "#1B998B", "#6C4AB6", "#5BC0EB", "#F49097", "#9AA5B1"]      # colores del curso
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=PALETA), "axes.titleweight": "bold", "figure.dpi": 110})
print("Herramientas listas.")

## 1. Formulación

Imagínense a alguien que vive de vender sancocho en Manizales. Compra en la Galería, cocina para seis por olla y tiene que imprimir la carta con el precio del plato. La carta dura un trimestre. Si el mercado sube y el precio se quedó quieto, el trimestre se lo come el espinazo.

Esa persona no necesita un modelo. Necesita tres respuestas:

1. **Predecir.** ¿Cuánto costará la olla dentro de tres meses?
2. **Explicar.** ¿Qué ingrediente mueve el costo?
3. **Decidir.** ¿Con qué costo debe presupuestar la olla del próximo trimestre para no quedar corta?

**El dato.** El DANE publica cada mes, en el sistema SIPSA, el precio mayorista por kilo de cientos de productos en las centrales de abasto del país. Manizales reporta por la Galería.

**La receta, para seis:** 1,5 kg de espinazo de cerdo, 1 kg de yuca, 1 kg de papa, tres mazorcas (0,9 kg) y cuatro plátanos verdes (1,4 kg).

**La métrica.** Medimos el error porcentual absoluto medio (MAPE) del pronóstico a tres meses. Un MAPE de 6 % significa que, en promedio, el pronóstico se aleja 6 % del costo real. Elegimos un porcentaje porque el dueño del negocio piensa en márgenes, no en pesos sueltos.

### Predicción escrita

Antes de ver un solo dato, escriban aquí su apuesta. Después la comparamos con lo que salga.

- ¿Cuánto costaban los ingredientes de la olla en diciembre de 2024, al por mayor? **Mi apuesta:**
- ¿Qué ingrediente pesa más en el costo? **Mi apuesta:**
- Si yo pronostico «dentro de tres meses costará lo mismo que hoy», ¿de qué tamaño será mi error? **Mi apuesta:**

## 2. Datos

Una sola línea trae el dato desde el DANE. La función descarga los ocho archivos oficiales, los unifica y revisa con una huella (MD5) que el resultado sea idéntico al que usé para preparar la clase. Tarda cerca de medio minuto.

El archivo no está guardado en el repositorio del curso porque el DANE no permite republicarlo. Por eso cada uno lo descarga de la fuente.

In [ ]:
sipsa = descargar("sipsa")
sipsa.head()

In [ ]:
print("Filas:", len(sipsa))
print("Meses:", sipsa["fecha"].min(), "a", sipsa["fecha"].max())
print("Ciudades:", sipsa["ciudad"].nunique(), "| Productos:", sipsa["producto"].nunique())

Nos quedamos con Manizales. Antes de filtrar productos conviene mirar cómo se llaman, porque el DANE no dice «papa»: dice la variedad.

In [ ]:
manizales = sipsa[sipsa["ciudad"] == "Manizales"]
print(manizales["mercado"].unique())
for palabra in ["espinazo", "Yuca", "Papa", "mazorca", "Plátano"]:
    print(palabra, "->", sorted(p for p in manizales["producto"].unique() if palabra.lower() in p.lower()))

Hay varias papas y varios plátanos. Escoger uno es una **decisión de modelación** y va a la bitácora de la sección 4. Tomo las variedades que se venden todos los meses en la Galería.

In [ ]:
receta = {                                  # kilos para seis personas
    "Carne de cerdo, espinazo": 1.5,
    "Yuca ICA": 1.0,
    "Papa capira": 1.0,
    "Chócolo mazorca": 0.9,
    "Plátano dominico hartón verde": 1.4,
}
precios = (manizales[manizales["producto"].isin(receta)]
           .pivot_table(index="fecha", columns="producto", values="precio_kg"))
precios.index = pd.to_datetime(precios.index)
print("Meses:", len(precios), "| Faltantes por producto:")
print(precios.isna().sum())

Ningún mes falta. El costo de la olla es la suma de precio por cantidad.

In [ ]:
costo_por_ingrediente = precios * pd.Series(receta)
olla = costo_por_ingrediente.sum(axis=1).rename("olla")

ax = olla.plot(color="#D7263D", lw=2)
ax.set(title="Costo mayorista de una olla de sancocho para seis, Manizales", xlabel="", ylabel="pesos")
plt.show()
print(f"Enero de 2013: ${olla.iloc[0]:,.0f} | Diciembre de 2024: ${olla.iloc[-1]:,.0f} | Se multiplicó por {olla.iloc[-1] / olla.iloc[0]:.2f}")

Comparen con su apuesta. Ojo con lo que mide esta cifra: es el costo **mayorista** de los ingredientes en pesos corrientes. No incluye gas, mano de obra ni el margen de la tienda.

### 2.1 Conocer el dato antes de modelarlo

Antes de pronosticar nada conviene mirar el dato por cuatro lados: cuánto vale cada ingrediente, cuánto se mueve de un mes a otro, si tiene temporada y cuánta memoria tiene. Llamo $p_{i,t}$ al precio por kilo del ingrediente $i$ en el mes $t$ y $q_i$ a los kilos de la receta. El costo de la olla es

$$C_t=\sum_{i=1}^{5} q_i\,p_{i,t},$$

y la variación mensual de cada precio es

$$r_{i,t}=\frac{p_{i,t}-p_{i,t-1}}{p_{i,t-1}}.$$

In [ ]:
ficha = pd.DataFrame({
    "kilos en la receta": pd.Series(receta),
    "precio en ene 2013": precios.iloc[0],
    "precio en dic 2024": precios.iloc[-1],
    "veces que subió": precios.iloc[-1] / precios.iloc[0],
    "mínimo": precios.min(),
    "mediana": precios.median(),
    "máximo": precios.max(),
    "meses sin dato": precios.isna().sum(),
})
display(ficha.style.format("{:,.0f}").format("{:.2f}", subset=["veces que subió"]).format("{:.1f}", subset=["kilos en la receta"])
        .background_gradient(subset=["veces que subió"], cmap="Blues").set_caption("Ficha del dato: pesos por kilo, Galería de Manizales"))

ejes = precios.plot(subplots=True, layout=(2, 3), figsize=(12, 5), legend=False, color="#1D4E89", lw=1.2)
for eje, nombre in zip(ejes.ravel(), precios.columns):
    eje.set_title(nombre, fontsize=9); eje.set_xlabel("")
ejes.ravel()[-1].axis("off")
plt.suptitle("Precio mayorista por kilo, Manizales"); plt.tight_layout(); plt.show()

In [ ]:
cambios = precios.pct_change().dropna()            # variación mensual de cada ingrediente

fig, (izq, der) = plt.subplots(1, 2, figsize=(12, 4))
(cambios * 100).plot.box(ax=izq, rot=20, color="#1D4E89")
izq.set(title="Variación mensual por ingrediente", ylabel="%"); izq.tick_params(axis="x", labelsize=7)

correlacion = cambios.corr()
imagen = der.imshow(correlacion, cmap="RdBu_r", vmin=-1, vmax=1)
der.set_xticks(range(5)); der.set_yticks(range(5))
der.set_xticklabels([c[:12] for c in correlacion.columns], rotation=30, fontsize=7); der.set_yticklabels([c[:12] for c in correlacion.columns], fontsize=7)
for a in range(5):
    for b in range(5):
        der.text(b, a, f"{correlacion.iloc[a, b]:.2f}", ha="center", va="center", fontsize=8)
der.set_title("Correlación entre variaciones mensuales"); plt.colorbar(imagen, ax=der); plt.tight_layout(); plt.show()

print("Desviación estándar de la variación mensual (%):")
print((cambios.std() * 100).round(1).sort_values(ascending=False))

Dos lecturas. La primera es que hay ingredientes tranquilos e ingredientes nerviosos, y el más nervioso no tiene por qué ser el que más pesa en la olla. La segunda es que las variaciones casi no están correlacionadas entre sí: cada ingrediente tiene su propia historia. Eso lo retomamos con el PCA.

**Temporada y memoria.** Para ver la temporada comparo cada mes con el promedio móvil de doce meses centrado en él. Para ver la memoria uso la autocorrelación de orden $k$:

$$\hat\rho_k=\frac{\sum_{t=k+1}^{T}(C_t-\bar C)(C_{t-k}-\bar C)}{\sum_{t=1}^{T}(C_t-\bar C)^2}.$$

In [ ]:
tendencia = olla.rolling(12, center=True).mean()
indice_estacional = (olla / tendencia).groupby(olla.index.month).mean()

fig, (izq, centro, der) = plt.subplots(1, 3, figsize=(13, 3.6))
((indice_estacional - 1) * 100).plot.bar(ax=izq, color="#1B998B")
izq.set(title="Temporada: desvío frente a la tendencia", xlabel="mes", ylabel="%")

rezagos = range(1, 25)
centro.bar(rezagos, [olla.autocorr(k) for k in rezagos], color="#1D4E89")
centro.set(title="Autocorrelación del costo", xlabel="rezago (meses)")
der.bar(rezagos, [olla.pct_change().autocorr(k) for k in rezagos], color="#D7263D")
der.axhline(2 / np.sqrt(len(olla)), color="gray", ls=":"); der.axhline(-2 / np.sqrt(len(olla)), color="gray", ls=":")
der.set(title="Autocorrelación de la variación mensual", xlabel="rezago (meses)")
plt.tight_layout(); plt.show()

La gráfica del centro dice que el costo de hoy se parece mucho al de los meses anteriores: la serie tiene memoria larga porque tiene tendencia. La de la derecha quita esa tendencia y pregunta algo más fino: si este mes subió, ¿el próximo también? Las líneas punteadas marcan la banda $\pm 2/\sqrt{T}$, dentro de la cual una autocorrelación no se distingue del ruido.

Guarden esta imagen. Las variaciones tienen poca memoria: solo el primer rezago se asoma con claridad por fuera de la banda. Una serie cuyas variaciones no tienen memoria se llama **caminata aleatoria**, $C_{t+1}=C_t+\varepsilon_{t+1}$, y para una caminata aleatoria el mejor pronóstico es el último valor. La olla se le parece bastante, y eso anticipa lo que veremos en la sección 3.

### 2.2 Una prueba de hipótesis: ¿es la olla una caminata aleatoria?

La gráfica sugiere que sí. Una prueba lo pone en números. La prueba de Dickey-Fuller aumentada ajusta la regresión

$$\Delta c_t=\alpha+\gamma\,c_{t-1}+\sum_{j=1}^{p}\phi_j\,\Delta c_{t-j}+\varepsilon_t,\qquad c_t=\log C_t,$$

y contrasta

$$H_0:\ \gamma=0\quad\text{(raíz unitaria: la serie no regresa a un nivel fijo)}\qquad\text{contra}\qquad H_1:\ \gamma<0.$$

Un valor $p$ pequeño rechaza $H_0$. La aplico dos veces: al nivel de la olla y a sus variaciones mensuales.

**Predicción escrita:** ¿en cuál de las dos se rechazará $H_0$? **Mi apuesta:**

In [ ]:
from statsmodels.tsa.stattools import adfuller

import warnings
log_olla = np.log(olla)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")              # la función avisa de un cambio futuro de formato que no nos afecta
    p_nivel = adfuller(log_olla)[1]
    p_variacion = adfuller(log_olla.diff().dropna())[1]
print(f"Valor p para el nivel de la olla:        {p_nivel:.3f}")
print(f"Valor p para las variaciones mensuales:  {p_variacion:.4f}")

En el nivel no se rechaza la raíz unitaria: la olla no tiene un precio «normal» al que regrese. En las variaciones sí se rechaza, así que los cambios mes a mes son estables. Es la firma de una caminata aleatoria con deriva. No rechazar $H_0$ no la demuestra, pero la deja como la descripción más económica del dato.

La consecuencia práctica es que la cantidad correcta para modelar es el **cambio** y no el nivel. Lo vamos a usar dos veces en la sección 3.

### 2.3 PCA, primera mirada: cada punto es un mes

El análisis de componentes principales busca las direcciones en que más varían los datos. Aquí cada mes es un punto con cinco coordenadas, que son los precios de los cinco ingredientes. Estandarizo los logaritmos de los precios en una matriz $Z$ y descompongo su matriz de correlaciones:

$$R=\frac{1}{T-1}Z^\top Z=V\Lambda V^\top,\qquad \Lambda=\operatorname{diag}(\lambda_1\ge\dots\ge\lambda_5).$$

Las columnas de $V$ son los componentes, $\lambda_j/\sum_k\lambda_k$ es la fracción de la varianza que explica el componente $j$, y las nuevas coordenadas de cada mes son los **puntajes** $ZV$. Con los tres primeros puntajes se dibuja cada mes como un punto en el espacio.

**Predicción escrita:** si dibujo los 144 meses en ese espacio y los coloreo por año, ¿qué figura esperan ver? **Mi apuesta:**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots

AZUL, ROJO, VERDE, MORADO, CIELO, ROSA, GRIS = PALETA
FUENTE = dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#1F2937")
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

Z = StandardScaler().fit_transform(np.log(precios))
pca = PCA().fit(Z)
puntajes = pca.transform(Z)
explicada = pca.explained_variance_ratio_ * 100
cargas = pd.DataFrame(pca.components_.T, index=precios.columns, columns=[f"CP{j + 1}" for j in range(5)])

resumen_pca = make_subplots(rows=1, cols=2, column_widths=[0.38, 0.62], horizontal_spacing=0.22,
                            subplot_titles=("Varianza explicada por componente", "Cargas: qué mezcla de ingredientes es cada componente"))
resumen_pca.add_trace(go.Bar(x=list(cargas.columns), y=explicada, marker_color=AZUL, text=[f"{v:.0f} %" for v in explicada],
                             textposition="outside", hovertemplate="%{x}: %{y:.1f} %<extra></extra>"), row=1, col=1)
resumen_pca.add_trace(go.Heatmap(z=cargas[["CP1", "CP2", "CP3"]].values, x=["CP1", "CP2", "CP3"], y=list(cargas.index), zmid=0,
                                 colorscale=[[0, ROJO], [0.5, "#FFFFFF"], [1, AZUL]], text=cargas[["CP1", "CP2", "CP3"]].round(2).values,
                                 texttemplate="%{text}", colorbar=dict(thickness=12, len=0.8), hovertemplate="%{y}<br>%{x}: %{z:.2f}<extra></extra>"), row=1, col=2)
resumen_pca.update_yaxes(title_text="%", range=[0, 85], row=1, col=1)
resumen_pca.update_layout(template="plotly_white", height=340, font=FUENTE, showlegend=False, margin=dict(l=10, r=10, t=50, b=10))
resumen_pca.show()

In [ ]:
anio_decimal = precios.index.year + (precios.index.month - 1) / 12
etiquetas = [f"<b>{MESES[fecha.month - 1]} de {fecha.year}</b><br>olla: ${costo:,.0f}" for fecha, costo in zip(precios.index, olla)]

camino = go.Figure()
camino.add_trace(go.Scatter3d(
    x=puntajes[:, 0], y=puntajes[:, 1], z=puntajes[:, 2], mode="lines+markers",
    line=dict(color="rgba(120, 130, 145, 0.55)", width=3),
    marker=dict(size=5, color=anio_decimal, colorscale=[[0, CIELO], [0.45, AZUL], [0.75, MORADO], [1, ROJO]],
                colorbar=dict(title="año", thickness=14, len=0.6, tickformat="d"), line=dict(color="white", width=0.5)),
    text=etiquetas, hovertemplate="%{text}<extra></extra>"))
largo = 3.2                                           # las flechas de las cargas se alargan para que se vean junto a los puntos
for ingrediente in cargas.index:
    v = cargas.loc[ingrediente, ["CP1", "CP2", "CP3"]].values * largo
    camino.add_trace(go.Scatter3d(x=[0, v[0]], y=[0, v[1]], z=[0, v[2]], mode="lines+text", line=dict(color=VERDE, width=6),
                                  text=["", ingrediente.split(",")[-1].strip()], textposition="top center",
                                  textfont=dict(size=11, color=VERDE), hoverinfo="skip"))
camino.update_layout(
    template="plotly_white", height=640, font=FUENTE, showlegend=False, margin=dict(l=0, r=0, t=60, b=0),
    title=dict(text="<b>Doce años de la olla en tres componentes</b><br><sup>Cada punto es un mes. Arrastren para girar, rueda para acercar, pasen el ratón por un punto.</sup>"),
    scene=dict(xaxis_title=f"CP1 ({explicada[0]:.0f} %)", yaxis_title=f"CP2 ({explicada[1]:.0f} %)", zaxis_title=f"CP3 ({explicada[2]:.0f} %)",
               camera=dict(eye=dict(x=1.5, y=-1.6, z=0.9))))
camino.show()

Giren la figura antes de leer. El primer componente explica la mayor parte, y en el mapa de cargas sus cinco números tienen el mismo signo: es el nivel general de precios, lo que uno llamaría la inflación de la olla. Por eso los meses forman un camino que avanza con los años a lo largo de ese eje. Hay un tramo donde los puntos se separan de golpe. Búsquenlo con el ratón y digan en qué meses cae.

El segundo componente tiene cargas de signos opuestos. Opone unos ingredientes a otros, así que recoge los meses en que la olla cambia de composición sin cambiar de precio. Las flechas verdes son las cargas: un mes que queda lejos en la dirección de una flecha es un mes en que ese ingrediente estaba caro frente a su propia historia.

### 2.4 PCA, segunda mirada: cada punto es un producto

Ahora le doy la vuelta a la tabla. Tomo todos los productos que la Galería reporta casi todos los meses, y cada **producto** es un punto cuyas coordenadas son sus variaciones mensuales. Dos productos quedan cerca si suben y bajan en los mismos meses.

La pregunta es si los alimentos forman familias, y dónde caen los cinco ingredientes del sancocho. La agrupación por familia que uso para colorear la hice yo a partir del nombre del producto. El PCA no la conoce.

In [ ]:
todos_los_precios = manizales.pivot_table(index="fecha", columns="producto", values="precio_kg")
frecuentes = todos_los_precios.loc[:, todos_los_precios.notna().sum() >= 140].interpolate().bfill().ffill()
variaciones = np.log(frecuentes).diff().dropna()

FRUTAS = ["aguacate", "banano", "coco", "fresa", "guanábana", "guayaba", "kiwi", "limón", "lulo", "mandarina", "mango", "manzana",
          "maracuyá", "melón", "mora", "naranja", "papaya", "pera", "piña", "tangelo", "tomate de árbol", "uva"]
VERDURAS = ["ahuyama", "ajo", "apio", "arveja verde en vaina", "berenjena", "brócoli", "cebolla", "chócolo", "cilantro", "coliflor",
            "espinaca", "fríjol verde", "habichuela", "lechuga", "pepino", "perejil", "pimentón", "remolacha", "repollo", "tomate chonto", "zanahoria"]
TUBERCULOS = ["papa ", "yuca", "plátano", "arracacha"]

def familia(nombre):
    n = nombre.lower()
    if n.startswith("carne de res"): return "carne de res"
    if n.startswith("carne de cerdo"): return "carne de cerdo"
    if n.startswith("huevo"): return "huevos"
    if any(n.startswith(x) for x in FRUTAS): return "frutas"
    if any(n.startswith(x) for x in TUBERCULOS): return "tubérculos y plátanos"
    if any(n.startswith(x) for x in VERDURAS): return "verduras y hortalizas"
    return "abarrotes y otros"

Zp = StandardScaler().fit_transform(variaciones)            # una columna por producto
pca_productos = PCA(n_components=3).fit(Zp.T)               # al trasponer, cada producto es una fila
nube = pd.DataFrame(pca_productos.transform(Zp.T), index=frecuentes.columns, columns=["CP1", "CP2", "CP3"])
nube["familia"] = [familia(p) for p in nube.index]
var_p = pca_productos.explained_variance_ratio_ * 100
print(f"{len(nube)} productos | varianza explicada por los tres primeros componentes (%):", var_p.round(1))
print(nube["familia"].value_counts().to_string())

In [ ]:
colores = {"carne de res": ROJO, "carne de cerdo": ROSA, "huevos": MORADO, "frutas": VERDE,
           "verduras y hortalizas": "#8BC34A", "tubérculos y plátanos": AZUL, "abarrotes y otros": GRIS}

galeria = go.Figure()
for nombre in colores:
    grupo = nube[nube["familia"] == nombre]
    galeria.add_trace(go.Scatter3d(
        x=grupo["CP1"], y=grupo["CP2"], z=grupo["CP3"], mode="markers", name=f"{nombre} ({len(grupo)})",
        marker=dict(size=6, color=colores[nombre], opacity=0.92, line=dict(color="white", width=0.6)),
        text=grupo.index, hovertemplate="<b>%{text}</b><extra>" + nombre + "</extra>"))
sancocho = nube.loc[list(receta)]
galeria.add_trace(go.Scatter3d(
    x=sancocho["CP1"], y=sancocho["CP2"], z=sancocho["CP3"], mode="markers+text", name="ingredientes del sancocho",
    marker=dict(size=11, symbol="diamond-open", color="#111827", line=dict(color="#111827", width=3)),
    text=[p.split(",")[-1].strip() for p in sancocho.index], textposition="top center", textfont=dict(size=12, color="#111827"),
    hovertemplate="<b>%{text}</b><extra>sancocho</extra>"))
galeria.update_layout(
    template="plotly_white", height=680, font=FUENTE, margin=dict(l=0, r=0, t=60, b=0),
    title=dict(text=f"<b>Los {len(nube)} productos de la Galería en tres componentes</b><br><sup>Cada punto es un producto. Un clic en la leyenda apaga una familia; dos clics la dejan sola.</sup>"),
    legend=dict(title="familia", itemsizing="constant", bgcolor="rgba(255,255,255,0.85)", x=0.01, y=0.98),
    scene=dict(xaxis_title=f"CP1 ({var_p[0]:.0f} %)", yaxis_title=f"CP2 ({var_p[1]:.0f} %)", zaxis_title=f"CP3 ({var_p[2]:.0f} %)",
               camera=dict(eye=dict(x=1.6, y=1.5, z=0.8))))
galeria.show()

Nadie le dijo al PCA qué es carne y qué es fruta, y aun así las familias aparecen. Giren la nube hasta que las carnes de res queden juntas en un extremo, con las de cerdo a su lado, y busquen los huevos. Apaguen familias en la leyenda para ver cada una sola. Los cinco ingredientes del sancocho, marcados con un rombo, caen en zonas distintas. Esa es la versión geométrica de lo que dijo la matriz de correlaciones: cada ingrediente tiene su propia historia, y por eso la olla promedia riesgos en lugar de sumarlos.

Dos advertencias. Los tres primeros componentes explican poca varianza, así que esta nube es una sombra de otra que vive en muchas dimensiones. Y la cercanía dice que dos productos se mueven parecido, sin decir por qué.

El PCA aquí sirve para describir. No lo vamos a usar como entrada del modelo.

## 3. Modelado

### 3.1 La línea base

Todo modelo se mide contra el pronóstico más barato posible. Con horizonte $h=3$ meses hay dos candidatos que no cuestan nada:

$$\hat C^{\text{ing}}_{t+h}=C_t\qquad\text{y}\qquad \hat C^{\text{est}}_{t+h}=C_{t+h-12}.$$

El primero dice que dentro de tres meses costará lo mismo que hoy. El segundo, que costará lo mismo que ese mes del año pasado. El error se mide con

$$\text{MAPE}=\frac{100}{n}\sum_{t}\left|\frac{C_{t+h}-\hat C_{t+h}}{C_{t+h}}\right|.$$

Si un modelo sofisticado no les gana, no sirve, por elegante que sea.

In [ ]:
H = 3                                             # horizonte: tres meses
tabla = pd.DataFrame({"real": olla.shift(-H)})    # lo que queremos adivinar: el costo en t+3
tabla["ingenuo"] = olla                           # lo que sabemos hoy
tabla["estacional"] = olla.shift(12 - H)          # el mismo mes objetivo, un año antes
tabla = tabla.dropna()

def mape(real, pronostico):
    return float((abs(real - pronostico) / real).mean() * 100)

for metodo in ["ingenuo", "estacional"]:
    print(f"MAPE {metodo}: {mape(tabla['real'], tabla[metodo]):.1f} %")

Ya hay una cifra que vencer. Comparen con su tercera apuesta.

### 3.2 La trampa de barajar

Empecemos por el modelo más simple, una regresión lineal sobre el costo de hoy, el de los dos meses anteriores y el del mismo mes objetivo un año atrás:

$$C_{t+h}=\beta_0+\beta_1C_t+\beta_2C_{t-1}+\beta_3C_{t-2}+\beta_4C_{t+h-12}+\varepsilon_{t+h}.$$

In [ ]:
X = pd.DataFrame({
    "hoy": olla,
    "hace_1": olla.shift(1),
    "hace_2": olla.shift(2),
    "anio_pasado": olla.shift(12 - H),
})
datos = X.join(olla.shift(-H).rename("real")).dropna()
variables = list(X.columns)
print(len(datos), "meses utilizables, de", datos.index.min().date(), "a", datos.index.max().date())

Ahora viene la decisión que más errores causa en la práctica: **cómo separar entrenamiento y prueba**. Hagámoslo de las dos maneras y comparemos.

- **Barajando:** se toma al azar el 25 % de los meses para probar. Es lo que hace `train_test_split` por defecto.
- **Respetando el tiempo:** se entrena con todo hasta diciembre de 2021 y se prueba con 2022 en adelante.

**Predicción escrita:** ¿cuál de las dos dará un error más pequeño, y por qué? **Mi apuesta:**

In [ ]:
from sklearn.model_selection import train_test_split

# Manera 1: barajando. Se repite 200 veces para que el resultado no dependa de un sorteo con suerte.
barajados = []
for semilla in range(200):
    X_a, X_b, y_a, y_b = train_test_split(datos[variables], datos["real"], test_size=0.25, random_state=semilla)
    barajados.append(mape(y_b, LinearRegression().fit(X_a, y_a).predict(X_b)))
mape_barajado = float(np.mean(barajados))

# Manera 2: respetando el tiempo
antes, despues = datos.loc[:"2021-12-01"], datos.loc["2022-01-01":]
mape_temporal = mape(despues["real"], LinearRegression().fit(antes[variables], antes["real"]).predict(despues[variables]))

print(f"MAPE barajando (promedio de 200 sorteos): {mape_barajado:.1f} %")
print(f"MAPE respetando el tiempo:                 {mape_temporal:.1f} %")

Los dos números miden el mismo modelo, así que la diferencia viene del examen. Al barajar, el modelo se entrena con meses de 2023 y lo examinan con meses de 2022: ya vio el futuro. Ese examen no se parece a lo que le pasará al dueño del negocio, que siempre pronostica hacia adelante. De aquí en adelante solo vale la validación que respeta el tiempo.

### 3.3 Cinco modelos y un protocolo para escoger

Una regresión es un candidato entre muchos. Vamos a poner a competir cinco familias:

| Modelo | Idea en una línea |
|---|---|
| Regresión lineal | $\hat y=\beta_0+x^\top\beta$, con $\beta$ que minimiza $\sum(y-\hat y)^2$ |
| Ridge | Lo mismo, con castigo a los coeficientes grandes: $\min_\beta\sum(y-\hat y)^2+\alpha\lVert\beta\rVert_2^2$ |
| $k$ vecinos | Promedia el resultado de los $k$ meses pasados más parecidos al de hoy |
| Bosque aleatorio | Promedia muchos árboles de decisión entrenados sobre remuestreos del dato |
| *Gradient boosting* | Suma árboles pequeños, cada uno corrigiendo el error del anterior |

Si escogemos al ganador mirando la prueba, la prueba deja de ser un examen limpio: de cinco candidatos, alguno saldrá bien por suerte. Por eso el tiempo se parte en **tres** tramos:

$$\underbrace{2013\ \dots\ 2019}_{\text{entrenamiento}}\ \Big|\ \underbrace{2020\ \dots\ 2021}_{\text{validación: se escoge}}\ \Big|\ \underbrace{2022\ \dots\ 2024}_{\text{prueba: se reporta}}$$

Hay un detalle fino. La fila del mes $t$ lleva como respuesta el costo de $t+h$. Para que el entrenamiento no use respuestas que caen dentro del tramo siguiente, solo entran las filas con $t+h$ anterior al corte.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def cinco_modelos():
    return {
        "lineal": LinearRegression(),
        "ridge": make_pipeline(StandardScaler(), Ridge(alpha=10)),
        "k vecinos": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
        "bosque": RandomForestRegressor(n_estimators=300, random_state=0),
        "boosting": GradientBoostingRegressor(random_state=0),
    }

objetivo = datos.index + pd.DateOffset(months=H)                 # el mes que cada fila intenta adivinar
E = objetivo <= pd.Timestamp("2019-12-01")                        # entrenamiento
V = (datos.index >= "2020-01-01") & (objetivo <= pd.Timestamp("2021-12-01"))   # validación
P = datos.index >= "2022-01-01"                                   # prueba
print(f"Entrenamiento: {E.sum()} meses | Validación: {V.sum()} | Prueba: {P.sum()}")

def competir(entradas, respuesta, a_pesos):
    """Mide cada modelo en validación; luego lo reentrena con entrenamiento y validación y lo mide en prueba."""
    errores, pronosticos = {}, {}
    for nombre, modelo in cinco_modelos().items():
        modelo.fit(entradas[E], respuesta[E])
        en_validacion = mape(datos["real"][V], a_pesos(modelo.predict(entradas[V]), V))
        modelo.fit(entradas[E | V], respuesta[E | V])
        pronosticos[nombre] = pd.Series(a_pesos(modelo.predict(entradas[P]), P), index=datos.index[P])
        errores[nombre] = {"validación": en_validacion, "prueba": mape(datos["real"][P], pronosticos[nombre])}
    return pd.DataFrame(errores).T, pd.DataFrame(pronosticos)

errores_nivel, pron_nivel = competir(datos[variables], datos["real"], lambda pred, tramo: pred)
bases = pd.DataFrame({m: {"validación": mape(datos["real"][V], tabla.loc[datos.index[V], m]),
                          "prueba": mape(datos["real"][P], tabla.loc[datos.index[P], m])} for m in ["ingenuo", "estacional"]}).T
display(pd.concat([bases, errores_nivel]).round(1))

In [ ]:
fechas = datos.index[P] + pd.DateOffset(months=H)
plt.figure(figsize=(11, 4))
plt.plot(fechas, datos["real"][P], color="black", lw=2.5, label="real")
plt.plot(fechas, tabla.loc[datos.index[P], "ingenuo"], color="#1B998B", lw=1.5, label="ingenuo")
for nombre in pron_nivel:
    plt.plot(fechas, pron_nivel[nombre], lw=1, ls="--", label=nombre)
plt.axhline(datos["real"][E | V].max(), color="gray", ls=":")
plt.text(fechas[0], datos["real"][E | V].max() * 1.005, "máximo visto antes de 2022", fontsize=8, color="gray")
plt.title("Pronóstico a tres meses en la prueba"); plt.ylabel("pesos"); plt.legend(fontsize=8, ncol=4); plt.show()

Miren las líneas del bosque, del *boosting* y de los vecinos contra la línea punteada. Un árbol predice promediando respuestas que ya vio, y un vecino también. Ninguno puede pronosticar un costo más alto que el máximo de su entrenamiento. La olla de 2022 a 2024 cuesta más que cualquier olla anterior, así que esos modelos se quedan pegados al techo. Los modelos lineales sí extrapolan.

Esto tiene arreglo, y el arreglo es una decisión de formulación. En vez de pedirles el **nivel** del costo, les pedimos el **cambio relativo**:

$$y_t=\frac{C_{t+h}}{C_t},\qquad \hat C_{t+h}=\hat y_t\,C_t.$$

El cambio relativo de 2023 se parece a los cambios relativos de 2015, aunque los precios sean otros. Las entradas también se dividen por el costo de hoy.

In [ ]:
relativas = datos[variables].div(datos["hoy"], axis=0).drop(columns="hoy")
relativas["mes_sen"] = np.sin(2 * np.pi * datos.index.month / 12)
relativas["mes_cos"] = np.cos(2 * np.pi * datos.index.month / 12)
razon = datos["real"] / datos["hoy"]

errores_cambio, pron_cambio = competir(relativas, razon, lambda pred, tramo: pred * datos["hoy"][tramo].values)

todos = pd.concat([bases,
                   errores_nivel.rename(index=lambda n: n + " (nivel)"),
                   errores_cambio.rename(index=lambda n: n + " (cambio)")])
todos.sort_values("validación").plot.barh(figsize=(9, 5), color=["#1D4E89", "#1B998B"])
plt.xlabel("MAPE (%)"); plt.title("Doce candidatos, y faltan dos: validación para escoger, prueba para reportar"); plt.gca().invert_yaxis(); plt.show()
display(todos.sort_values("validación").round(1))

### 3.4 Modelos hechos para series de tiempo

Los cinco modelos anteriores no saben que los datos están ordenados: reciben una fila y devuelven un número. Hay modelos que parten del orden. Probamos dos clásicos.

**Holt-Winters** descompone la serie en nivel $\ell_t$, tendencia $b_t$ y estación $s_t$, y actualiza cada parte con un suavizado exponencial. Su pronóstico a $h$ pasos es

$$\hat c_{t+h}=\ell_t+(\phi+\phi^2+\dots+\phi^h)\,b_t+s_{t+h-12},$$

donde $\phi<1$ amortigua la tendencia para que no se dispare.

**SARIMA** modela las variaciones. Con $B$ el operador de rezago, $Bc_t=c_{t-1}$, el modelo $(1,1,1)(0,1,1)_{12}$ es

$$(1-\varphi B)(1-B)(1-B^{12})\,c_t=(1+\theta B)(1+\Theta B^{12})\,\varepsilon_t.$$

El factor $(1-B)$ toma la diferencia mensual, que es lo que la prueba de la sección 2.2 pidió. El factor $(1-B^{12})$ quita la estación.

Estos modelos se evalúan con **origen móvil**: para pronosticar desde el mes $t$ se ajustan solo con los datos hasta $t$, y así mes a mes. Es más exigente con el computador y más fiel a como se usarían. La celda tarda cerca de un minuto.

In [ ]:
import warnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

def holt_winters(serie, pasos=H):
    ajuste = ExponentialSmoothing(np.log(serie), trend="add", damped_trend=True, seasonal="add", seasonal_periods=12).fit()
    return float(np.exp(ajuste.forecast(pasos).iloc[-1]))

def sarima(serie, pasos=H):
    ajuste = SARIMAX(np.log(serie), order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    return float(np.exp(ajuste.forecast(pasos).iloc[-1]))

series_de_tiempo = {"Holt-Winters": holt_winters, "SARIMA": sarima}
olla_m = olla.asfreq("MS")
pron_series, filas = {}, {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for nombre, metodo in series_de_tiempo.items():
        desde_cada_mes = pd.Series({t: metodo(olla_m[:t]) for t in datos.index[V | P]})   # origen móvil
        pron_series[nombre] = desde_cada_mes[datos.index[P]]
        filas[nombre] = {"validación": mape(datos["real"][V], desde_cada_mes[datos.index[V]]),
                         "prueba": mape(datos["real"][P], desde_cada_mes[datos.index[P]])}
pron_series = pd.DataFrame(pron_series)
todos = pd.concat([todos, pd.DataFrame(filas).T])
display(todos.loc[["ingenuo", "estacional", "Holt-Winters", "SARIMA"]].round(1))

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    completo = SARIMAX(np.log(olla_m), order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    futuro = completo.get_forecast(6)
centro = np.exp(futuro.predicted_mean)
banda = np.exp(futuro.conf_int(alpha=0.20))

plt.figure(figsize=(11, 4))
plt.plot(olla_m["2021":], color="#1D4E89", lw=2, label="costo observado")
plt.plot(np.exp(completo.fittedvalues)["2021":], color="#1B998B", lw=1, ls="--", label="SARIMA, un paso adelante")
plt.plot(centro, color="#D7263D", lw=2, label="pronóstico a seis meses")
plt.fill_between(banda.index, banda.iloc[:, 0], banda.iloc[:, 1], color="#D7263D", alpha=0.18, label="franja de 80 %")
plt.title("El modelo sigue la curva hacia atrás y abre un abanico hacia adelante"); plt.ylabel("pesos"); plt.legend(fontsize=8); plt.show()

Esta gráfica tiene una trampa visual que conviene nombrar. La línea punteada va pegada a la curva observada, y uno diría que el modelo es excelente. Pero cada punto de esa línea se calculó conociendo el mes anterior. El pronóstico ingenuo dibujaría una línea igual de pegada, corrida un mes. **Seguir la curva hacia atrás es fácil. Lo que cuesta es el abanico rojo**, que es donde el modelo ya no ve el dato. Miren qué tan ancho se pone en seis meses.

### 3.5 ¿Y una red neuronal?

Ustedes vieron redes recurrentes en Aprendizaje Profundo, así que la pregunta es natural. Una LSTM recibe las doce últimas variaciones mensuales y predice el cambio de los tres meses siguientes.

**Predicción escrita:** ¿le ganará la red al pronóstico ingenuo? ¿Con cuántas filas se entrena? **Mi apuesta:**

In [ ]:
import torch
from torch import nn

variacion = np.log(olla).diff().dropna()
ventanas, respuestas, origenes = [], [], []
for i in range(12, len(variacion) - H + 1):
    ventanas.append(variacion.values[i - 12:i]); respuestas.append(variacion.values[i:i + H].sum()); origenes.append(variacion.index[i - 1])
ventanas = torch.tensor(np.array(ventanas), dtype=torch.float32).unsqueeze(-1) * 10     # se escala para que la red entrene mejor
respuestas = torch.tensor(np.array(respuestas), dtype=torch.float32) * 10
origenes = pd.DatetimeIndex(origenes); destino_red = origenes + pd.DateOffset(months=H)
E_red = destino_red <= pd.Timestamp("2019-12-01")
V_red = (origenes >= "2020-01-01") & (destino_red <= pd.Timestamp("2021-12-01"))
P_red = origenes >= "2022-01-01"

class Recurrente(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm, self.salida = nn.LSTM(1, 16, batch_first=True), nn.Linear(16, 1)
    def forward(self, x):
        return self.salida(self.lstm(x)[0][:, -1]).squeeze(-1)

resultados = []
for semilla in range(5):
    torch.manual_seed(semilla)
    red = Recurrente(); optimizador = torch.optim.Adam(red.parameters(), lr=0.01, weight_decay=1e-3)
    for _ in range(300):
        optimizador.zero_grad(); perdida = ((red(ventanas[E_red]) - respuestas[E_red]) ** 2).mean(); perdida.backward(); optimizador.step()
    with torch.no_grad():
        pronostico_red = olla.reindex(origenes).values * np.exp(red(ventanas).numpy() / 10)
    real_red = olla.reindex(destino_red).values
    resultados.append({"semilla": semilla, "validación": mape(real_red[V_red], pronostico_red[V_red]), "prueba": mape(real_red[P_red], pronostico_red[P_red])})
resultados = pd.DataFrame(resultados).set_index("semilla")
print(f"Filas de entrenamiento: {int(E_red.sum())} | parámetros de la red: {sum(p.numel() for p in red.parameters())}")
display(resultados.round(1))
print(f"MAPE promedio de la red en prueba: {resultados['prueba'].mean():.1f} % | ingenuo: {todos.loc['ingenuo', 'prueba']:.1f} %")

Comparen el número de filas de entrenamiento con el número de parámetros de la red. Hay muchos más parámetros que datos. La red pierde contra el pronóstico ingenuo, y además cambia de resultado con la semilla, que es otra señal de que no hay dato suficiente para fijarla.

La lección no es que las redes sean malas. Una red recurrente o un *transformer* necesitan miles de series o series muy largas. Con 144 meses de una sola serie, un modelo con tres parámetros como SARIMA tiene ventaja porque trae la estructura puesta de antemano. La red no entra a la competencia por eso: dejarla concursar con cinco semillas sería comprar cinco boletas.

### 3.6 El veredicto

La regla quedó fijada antes de mirar: gana quien tenga menor MAPE en **validación**, y su error en **prueba** es el que se le reporta al cliente.

In [ ]:
mejor = todos["validación"].idxmin()
resumen = todos["prueba"].round(1)
print(f"Ganador en validación: {mejor}")
print(f"Su MAPE en prueba: {resumen[mejor]} %   |   MAPE del ingenuo en prueba: {resumen['ingenuo']} %")

gana_dos_veces = todos[(todos["validación"] < todos.loc["ingenuo", "validación"]) & (todos["prueba"] < todos.loc["ingenuo", "prueba"])]
elegido = gana_dos_veces["validación"].idxmin() if len(gana_dos_veces) else "ingenuo"
print("\nModelos que le ganan al ingenuo en validación y también en prueba:")
display(gana_dos_veces.sort_values("validación").round(1))
print(f"Modelo elegido: {elegido}")

pronosticos_prueba = pd.concat([tabla.loc[datos.index[P], ["ingenuo", "estacional"]],
                                pron_nivel.rename(columns=lambda n: n + " (nivel)"),
                                pron_cambio.rename(columns=lambda n: n + " (cambio)"), pron_series], axis=1)
plt.figure(figsize=(11, 3.8))
plt.plot(fechas, datos["real"][P], color="black", lw=2.5, label="real")
plt.plot(fechas, pronosticos_prueba["ingenuo"], color="#1B998B", label="ingenuo")
if mejor != "ingenuo":
    plt.plot(fechas, pronosticos_prueba[mejor], color="#D7263D", ls="--", lw=1, label=f"{mejor}: ganó la validación")
if elegido not in ("ingenuo", mejor):
    plt.plot(fechas, pronosticos_prueba[elegido], color="#6C4AB6", lw=2, label=f"{elegido}: gana en los dos tramos")
plt.title("La línea base, el ganador de validación y el modelo elegido"); plt.ylabel("pesos"); plt.legend(fontsize=8); plt.show()

Lean el resultado con calma. La regla que fijamos antes de mirar escogió un modelo que después perdió en la prueba contra el pronóstico ingenuo. Eso no es un error del procedimiento: es el procedimiento mostrando que un ganador de validación puede serlo por suerte, sobre todo con veintiún meses.

Aquí hago una segunda lectura, y la declaro como lo que es, una decisión de analista. La tabla de arriba deja solo los modelos que le ganan al ingenuo **en los dos tramos**. Un modelo que gana dos veces, en periodos distintos, es mejor candidato que uno que brilló una sola vez. De esos me quedo con el de menor error en validación.

Esta decisión tiene un costo y hay que decirlo. Usé la prueba para decidir, así que el error de prueba del modelo elegido deja de ser un examen limpio y hay que leerlo como optimista. Por eso al cliente no se le promete ese número. Se le promete algo más modesto: un modelo que se ha portado un poco mejor que repetir el último dato, y que además sabe decir cuánto puede fallar.

**¿La diferencia es real?** Dos MAPE separados por medio punto pueden ser la misma cosa. Para cada mes de la prueba tengo el error porcentual absoluto de un candidato, $a_t$, y el del ingenuo, $b_t$. La prueba de rangos con signo de Wilcoxon contrasta

$$H_0:\ \text{la mediana de } a_t-b_t \text{ es } 0\qquad\text{contra}\qquad H_1:\ \text{no lo es}.$$

In [ ]:
from scipy.stats import wilcoxon

error = lambda metodo: (datos["real"][P] - pronosticos_prueba[metodo]).abs() / datos["real"][P] * 100
filas_prueba = {}
for candidato in dict.fromkeys([mejor, "SARIMA", "Holt-Winters", "lineal (cambio)"]):
    if candidato != "ingenuo":
        filas_prueba[candidato] = {"MAPE en prueba": error(candidato).mean(), "MAPE del ingenuo": error("ingenuo").mean(),
                                   "valor p": wilcoxon(error(candidato), error("ingenuo")).pvalue}
display(pd.DataFrame(filas_prueba).T.round(3))
print(f"Meses en la prueba: {int(P.sum())}")

Lean la columna del valor $p$ con el número de meses al lado. Con tan pocos meses, una ventaja de medio punto no alcanza a distinguirse del azar. La prueba además es generosa, porque los errores de meses vecinos se parecen entre sí y eso hace ver más evidencia de la que hay. La versión cuidadosa de este contraste se llama prueba de Diebold-Mariano.

La conclusión honesta tiene dos partes. Varios modelos quedan por debajo del ingenuo en la prueba, y entre ellos están los dos de series de tiempo. Pero el dato no alcanza para declarar ganador a ninguno.

### 3.7 Explicar: ¿qué ingrediente mueve el costo?

Como el costo es una suma, el alza total se reparte de forma exacta entre ingredientes:

$$C_T-C_1=\sum_{i=1}^{5}q_i\,(p_{i,T}-p_{i,1}).$$

In [ ]:
participacion = (costo_por_ingrediente.iloc[-1] / olla.iloc[-1] * 100).round(1).sort_values(ascending=False)
print("Participación en el costo, diciembre de 2024 (%):")
print(participacion)

alza = costo_por_ingrediente.iloc[-1] - costo_por_ingrediente.iloc[0]
print("\nDe cada 100 pesos que subió la olla entre 2013 y 2024, cuántos puso cada ingrediente:")
print((alza / alza.sum() * 100).round(1).sort_values(ascending=False))

costo_por_ingrediente.plot.area(lw=0, color=PALETA[:5], alpha=0.9)
plt.title("Costo de la olla por ingrediente"); plt.xlabel(""); plt.ylabel("pesos"); plt.legend(fontsize=8); plt.show()

Esta explicación es una descomposición contable: dice cuánto pesa cada ingrediente, no por qué subió su precio. Para el porqué harían falta otros datos, como el precio del maíz que come el cerdo o el clima de la cosecha.

## 4. Comunicación

### 4.1 Decidir: ¿con cuánto se presupuesta la olla?

Un pronóstico de un solo número deja al cliente sin saber cuánto puede fallar. Usamos los errores de la prueba para ponerle un margen. Sea $u_t=C_{t+h}/\hat C_{t+h}-1$ lo que el costo real superó al pronóstico. El margen es el cuantil 0,90 de esos valores, y el costo para presupuestar es

$$\hat C_{T+h}\,\bigl(1+\hat Q_{0{,}90}(u)\bigr).$$

Usamos el modelo elegido en la sección 3.6. El margen sale de sus propios errores en la prueba.

In [ ]:
u = datos["real"][P] / pronosticos_prueba[elegido] - 1
margen = u.quantile(0.90)

hoy = olla.index[-1]
ultima = X.iloc[[-1]]
if elegido == "ingenuo":
    punto = olla.iloc[-1]
elif elegido == "estacional":
    punto = olla.shift(12 - H).iloc[-1]
elif elegido in series_de_tiempo:
    punto = series_de_tiempo[elegido](olla_m)
else:
    nombre, tipo = elegido.rsplit(" (", 1)
    modelo = cinco_modelos()[nombre]
    if tipo.startswith("nivel"):
        punto = float(modelo.fit(datos[variables], datos["real"]).predict(ultima[variables])[0])
    else:
        ultima_rel = ultima[variables].div(ultima["hoy"], axis=0).drop(columns="hoy")
        ultima_rel["mes_sen"] = np.sin(2 * np.pi * hoy.month / 12); ultima_rel["mes_cos"] = np.cos(2 * np.pi * hoy.month / 12)
        punto = float(modelo.fit(relativas, razon).predict(ultima_rel)[0]) * olla.iloc[-1]

destino_mes = hoy + pd.DateOffset(months=H)
print(f"Método elegido: {elegido} (MAPE en prueba {resumen[elegido]} %)")
print(f"Pronóstico para {MESES[destino_mes.month - 1]} de {destino_mes.year}: ${punto:,.0f}")
print(f"En 9 de cada 10 meses de prueba, el costo real no superó al pronóstico en más de {margen * 100:.1f} %")
print(f"Costo para presupuestar: ${punto * (1 + margen):,.0f}")

### La decisión en tres frases

Escríbanla para alguien que no sabe qué es una regresión. Una frase para lo que recomiendan, una para la evidencia y una para lo que puede salir mal.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 4.2 Bitácora de decisiones

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| Variedades de la receta (papa capira, yuca ICA, plátano dominico hartón) | Otras variedades | Se reportan todos los meses en la Galería |
| Horizonte de tres meses | Un mes, seis meses | Es lo que dura la carta impresa |
| MAPE como métrica | Error en pesos | El cliente piensa en márgenes |
| Tres tramos temporales: hasta 2019, 2020 a 2021 y 2022 a 2024 | Separación al azar; escoger mirando la prueba | Al barajar, el modelo ve el futuro; si se escoge en la prueba, la prueba deja de ser examen |
| Pronosticar el cambio relativo además del nivel | Solo el nivel | Árboles y vecinos no extrapolan por encima de lo visto |
| Margen con el percentil 90 de los meses de prueba | Intervalo teórico | Sale de errores observados y se explica sin fórmulas |
| Modelo final: el que le gana al ingenuo en validación y en prueba | El ganador de validación a secas | Ganó una vez y perdió la otra. El costo es que la prueba dejó de ser un examen limpio |
| La red neuronal no concursa | Incluirla entre los candidatos | 69 filas de entrenamiento; cinco semillas serían cinco boletas |
| Familias de productos por nombre, para colorear el PCA | Sin colorear | Ayuda a leer la nube; el PCA no usa la familia |

### 4.3 Riesgos y mantenimiento

Respondan en dos o tres líneas cada una.

- El dato llega hasta diciembre de 2024. ¿Qué habría que hacer para usar esto hoy?
- El precio es mayorista. ¿En qué se equivoca el cliente si lo lee como el precio de la tienda?
- ¿Cada cuánto habría que revisar si el pronóstico ingenuo sigue siendo el mejor?

### 4.4 Procedencia

Lo último que hace un cuaderno serio es decir de dónde salió su dato.

In [ ]:
procedencia()